In [ ]:
# Task 1: Encrypted Messaging App
# Goal: Simulate encrypted communication between two users using hybrid encryption (RSA + AES).

In [4]:
import os
from cryptography.hazmat.primitives.asymmetric import rsa, padding as asym_padding
from cryptography.hazmat.primitives import serialization, hashes
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives import padding as sym_padding
from cryptography.hazmat.backends import default_backend

# Update your base save path
BASE_PATH = r"C:\Users\JONI"

# --- RSA Key Generation ---
def generate_rsa_keys():
    private_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
    public_key = private_key.public_key()
    
    with open(os.path.join(BASE_PATH, "private_key.pem"), "wb") as f:
        f.write(private_key.private_bytes(
            encoding=serialization.Encoding.PEM,
            format=serialization.PrivateFormat.PKCS8,
            encryption_algorithm=serialization.NoEncryption()
        ))

    with open(os.path.join(BASE_PATH, "public_key.pem"), "wb") as f:
        f.write(public_key.public_bytes(
            encoding=serialization.Encoding.PEM,
            format=serialization.PublicFormat.SubjectPublicKeyInfo
        ))
    
    return private_key, public_key

# --- AES Encryption ---
def aes_encrypt(data, key, iv):
    padder = sym_padding.PKCS7(128).padder()
    padded_data = padder.update(data) + padder.finalize()
    cipher = Cipher(algorithms.AES(key), modes.CBC(iv), backend=default_backend())
    encryptor = cipher.encryptor()
    return encryptor.update(padded_data) + encryptor.finalize()

# --- AES Decryption ---
def aes_decrypt(ciphertext, key, iv):
    cipher = Cipher(algorithms.AES(key), modes.CBC(iv), backend=default_backend())
    decryptor = cipher.decryptor()
    padded_data = decryptor.update(ciphertext) + decryptor.finalize()
    unpadder = sym_padding.PKCS7(128).unpadder()
    return unpadder.update(padded_data) + unpadder.finalize()

# --- RSA Encryption ---
def rsa_encrypt(public_key, data):
    return public_key.encrypt(
        data,
        asym_padding.OAEP(
            mgf=asym_padding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None
        )
    )

# --- RSA Decryption ---
def rsa_decrypt(private_key, ciphertext):
    return private_key.decrypt(
        ciphertext,
        asym_padding.OAEP(
            mgf=asym_padding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None
        )
    )

# --- Main Task 1 Execution ---
def task1_execute():
    if not os.path.exists(BASE_PATH):
        os.makedirs(BASE_PATH)
    
    # 1. Generate RSA Keys
    private_key, public_key = generate_rsa_keys()
    
    # 2. Create the secret message
    with open(os.path.join(BASE_PATH, "message.txt"), "w") as f:
        f.write("This is a secret message from User B to User A.")
    
    # 3. Generate AES key and IV
    aes_key = os.urandom(32)  # 256-bit AES key
    iv = os.urandom(16)       # 128-bit IV
    
    # 4. Encrypt the message with AES
    with open(os.path.join(BASE_PATH, "message.txt"), "rb") as f:
        plaintext = f.read()
    
    ciphertext = aes_encrypt(plaintext, aes_key, iv)
    
    with open(os.path.join(BASE_PATH, "encrypted_message.bin"), "wb") as f:
        f.write(iv + ciphertext)
    
    # 5. Encrypt AES key with RSA public key
    encrypted_key = rsa_encrypt(public_key, aes_key)
    
    with open(os.path.join(BASE_PATH, "aes_key_encrypted.bin"), "wb") as f:
        f.write(encrypted_key)
    
    # 6. Decrypt AES key
    with open(os.path.join(BASE_PATH, "aes_key_encrypted.bin"), "rb") as f:
        encrypted_key = f.read()
    
    decrypted_aes_key = rsa_decrypt(private_key, encrypted_key)
    
    # 7. Decrypt the AES-encrypted message
    with open(os.path.join(BASE_PATH, "encrypted_message.bin"), "rb") as f:
        iv = f.read(16)
        encrypted_message = f.read()
    
    decrypted_message = aes_decrypt(encrypted_message, decrypted_aes_key, iv)
    
    # 8. Save the decrypted message
    with open(os.path.join(BASE_PATH, "decrypted_message.txt"), "wb") as f:
        f.write(decrypted_message)


# Run the Task 1 Script
if __name__ == "__main__":
    task1_execute()
    print("Task 1 completed! Files saved at C:\\Users\\JONI")


Task 1 completed! Files saved at C:\Users\JONI


In [6]:
# Task 2: Secure File Exchange (RSA + AES)
# Demonstrate a secure file transfer using hybrid encryption: 
# RSA (asymmetric encryption) to securely exchange the AES key.
# AES-256 (symmetric encryption) to encrypt the actual file contents efficiently.

# Hybrid encryption improves security without sacrificing speed or performance. The decrypted message exactly matches the original, and SHA-256 hash comparison confirms full integrity.


In [8]:
import os
import hashlib
from cryptography.hazmat.primitives.asymmetric import rsa, padding as asym_padding
from cryptography.hazmat.primitives import serialization, hashes
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives import padding as sym_padding
from cryptography.hazmat.backends import default_backend

# Set base path
BASE_PATH = r"C:\Users\JONI\Task 2"

# Helper functions
def aes_encrypt(data, key, iv):
    padder = sym_padding.PKCS7(128).padder()
    padded_data = padder.update(data) + padder.finalize()
    cipher = Cipher(algorithms.AES(key), modes.CBC(iv), backend=default_backend())
    encryptor = cipher.encryptor()
    return encryptor.update(padded_data) + encryptor.finalize()

def aes_decrypt(ciphertext, key, iv):
    cipher = Cipher(algorithms.AES(key), modes.CBC(iv), backend=default_backend())
    decryptor = cipher.decryptor()
    padded_data = decryptor.update(ciphertext) + decryptor.finalize()
    unpadder = sym_padding.PKCS7(128).unpadder()
    return unpadder.update(padded_data) + unpadder.finalize()

def rsa_encrypt(public_key, data):
    return public_key.encrypt(
        data,
        asym_padding.OAEP(
            mgf=asym_padding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None
        )
    )

def rsa_decrypt(private_key, ciphertext):
    return private_key.decrypt(
        ciphertext,
        asym_padding.OAEP(
            mgf=asym_padding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None
        )
    )

def compute_sha256(filepath):
    sha256 = hashlib.sha256()
    with open(filepath, "rb") as f:
        sha256.update(f.read())
    return sha256.hexdigest()

# Task 2 Execution
def task2_execute():
    if not os.path.exists(BASE_PATH):
        os.makedirs(BASE_PATH)
    
    # Generate RSA keys for Bob
    private_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
    public_key = private_key.public_key()

    with open(os.path.join(BASE_PATH, "private.pem"), "wb") as f:
        f.write(private_key.private_bytes(
            encoding=serialization.Encoding.PEM,
            format=serialization.PrivateFormat.PKCS8,
            encryption_algorithm=serialization.NoEncryption()
        ))

    with open(os.path.join(BASE_PATH, "public.pem"), "wb") as f:
        f.write(public_key.public_bytes(
            encoding=serialization.Encoding.PEM,
            format=serialization.PublicFormat.SubjectPublicKeyInfo
        ))

    # Alice creates a secret message
    with open(os.path.join(BASE_PATH, "alice_message.txt"), "w") as f:
        f.write("This is a top-secret file Alice wants to securely send to Bob.")

    # Generate AES key and IV
    aes_key = os.urandom(32)
    iv = os.urandom(16)

    # Encrypt Alice's message
    with open(os.path.join(BASE_PATH, "alice_message.txt"), "rb") as f:
        plaintext = f.read()

    ciphertext = aes_encrypt(plaintext, aes_key, iv)

    with open(os.path.join(BASE_PATH, "encrypted_file.bin"), "wb") as f:
        f.write(iv + ciphertext)

    # Encrypt AES key with RSA public key
    encrypted_aes_key = rsa_encrypt(public_key, aes_key)

    with open(os.path.join(BASE_PATH, "aes_key_encrypted.bin"), "wb") as f:
        f.write(encrypted_aes_key)

    # Bob decrypts the AES key
    with open(os.path.join(BASE_PATH, "aes_key_encrypted.bin"), "rb") as f:
        encrypted_key = f.read()

    decrypted_aes_key = rsa_decrypt(private_key, encrypted_key)

    # Bob decrypts the file
    with open(os.path.join(BASE_PATH, "encrypted_file.bin"), "rb") as f:
        iv = f.read(16)
        encrypted_message = f.read()

    decrypted_content = aes_decrypt(encrypted_message, decrypted_aes_key, iv)

    with open(os.path.join(BASE_PATH, "decrypted_message.txt"), "wb") as f:
        f.write(decrypted_content)

    # Verify integrity
    original_hash = compute_sha256(os.path.join(BASE_PATH, "alice_message.txt"))
    decrypted_hash = compute_sha256(os.path.join(BASE_PATH, "decrypted_message.txt"))

    if original_hash == decrypted_hash:
        print("Integrity Check Passed! Files match perfectly.")
    else:
        print("Integrity Check Failed! Files are different.")

In [10]:
# Run Task 2
if __name__ == "__main__":
    task2_execute()


Integrity Check Passed! Files match perfectly.


In [12]:
# Task3 - tls_summary.txt

In [14]:
import os

# Define base folder
BASE_PATH = r"C:\Users\JONI\Task 3"
os.makedirs(BASE_PATH, exist_ok=True)

# Define the summary content
tls_summary_content = """
TLS Communication Inspection Summary

1. Connection Details
- Server Connected: google.com
- TLS Version: TLS 1.3
- Cipher Suite Used: TLS_AES_256_GCM_SHA384
- Certificate Chain:
  - Root Certificate Authority (CA): GlobalSign Root CA
  - Intermediate Certificate Authority: Google Internet Authority G4
  - Leaf Certificate: *.google.com

2. Wireshark TLS Handshake Analysis
- Client Hello:
  - The client (browser) proposes supported TLS versions, cipher suites, and random numbers for secure negotiation.
- Server Certificate:
  - The server sends its public X.509 certificate to prove its identity.
- Key Exchange:
  - Ephemeral Diffie-Hellman (ECDHE) key exchange is performed to securely create a shared secret between the client and server.

3. How TLS Provides Confidentiality, Integrity, and Authentication
- Confidentiality:
  - After key exchange, all application data (HTTP) is encrypted using symmetric encryption (AES-256-GCM). This prevents anyone from reading the communication.

- Integrity:
  - TLS ensures that transmitted messages are not altered. This is achieved through authenticated encryption (AEAD) that combines encryption and integrity checking.

- Authentication:
  - The server’s certificate chain is verified by the client. This confirms that the server is trusted and prevents man-in-the-middle attacks.

Conclusion:
TLS ensures that communications over the internet are encrypted, verified, and protected against tampering. The handshake steps create a secure session where confidentiality, integrity, and authentication are guaranteed.
"""

# Write to tls_summary.txt
with open(os.path.join(BASE_PATH, "tls_summary.txt"), "w") as f:
    f.write(tls_summary_content.strip())

print(f"tls_summary.txt created successfully at {BASE_PATH}")


tls_summary.txt created successfully at C:\Users\JONI\Task 3


In [18]:
# Task 4 — Email Encryption and Signature Simulation

In [20]:
# Task 5 — Hashing & Integrity Check Utility

In [30]:
# Created a new Python file hash_util.py with the following code:

In [34]:
import hashlib
import json

def compute_hashes(filepath):
    with open(filepath, 'rb') as f:
        data = f.read()
    return {
        'sha256': hashlib.sha256(data).hexdigest(),
        'sha1': hashlib.sha1(data).hexdigest(),
        'md5': hashlib.md5(data).hexdigest()
    }

def save_hashes(filename, hashes):
    with open('hashes.json', 'w') as f:
        json.dump({filename: hashes}, f, indent=4)

def check_integrity(original_file, tampered_file):
    orig_hashes = compute_hashes(original_file)
    tampered_hashes = compute_hashes(tampered_file)

    print("\nIntegrity Check Results:")
    for algo in orig_hashes:
        if orig_hashes[algo] == tampered_hashes[algo]:
            print(f"[{algo.upper()}] PASS")
        else:
            print(f"[{algo.upper()}] FAIL")

# Save original file hashes
hashes = compute_hashes('original.txt')
save_hashes('original.txt', hashes)

# Simulate tampering by creating tampered.txt
with open('tampered.txt', 'w') as f:
    f.write("This file has been modified!")  # Change content to simulate tampering

# Check integrity
check_integrity('original.txt', 'tampered.txt')



Integrity Check Results:
[SHA256] FAIL
[SHA1] FAIL
[MD5] FAIL
